In [1]:
# uv add pyannote.audio numpy==1.26

In [2]:
import os
import torch
from dotenv import load_dotenv

load_dotenv()

HUGGING_FACE_TOKEN = os.getenv("HUGGING_FACE_TOKEN")

In [3]:
import torch
from pyannote.audio import Pipeline

# 파이프라인 인스턴스 생성 (최신 문법인 token 사용)
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HUGGING_FACE_TOKEN
)

# cuda가 사용 가능한 경우 cuda를 사용하도록 설정
if torch.cuda.is_available():
    pipeline.to(torch.device("cuda"))
    print('cuda is available')
else:
    print('cuda is not available')

c:\Users\user\Documents\GitHub\STUDY\chaewony\ch05\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
c:\Users\user\Documents\GitHub\STUDY\chaewony\ch05\.venv\Lib\site-packages\pyannote\audio\core\io.py:48: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

[WinError 127] 지정된 프로시저를 찾을 수 없습니다
  warnings.warn(


cuda is available


1. TqdmWarning: IProgress not found
원인: 주피터 노트북에서 예쁜 그래픽으로 작업 진행률 바를 보여주는 ipywidgets 패키지가 설치되어 있지 않아서 발생하는 경고입니다.

대처법: 진행률 바가 조금 투박한 텍스트 형태로 출력될 뿐, 코드 실행 결과에는 전혀 지장이 없으므로 무시하셔도 됩니다.

2. torchcodec UserWarning 및 [WinError 127]
원인: 최신 버전의 pyannote.audio는 음성 파일(mp3, wav 등)을 읽어 들일 때 torchcodec이라는 도구를 사용합니다. 하지만 윈도우 환경에서는 이 도구가 의존하는 내부 파일(DLL)을 찾지 못해 에러(WinError 127)가 자주 발생합니다.

대처법: 나중에 실제로 오디오 파일을 분석할 때, 파일 경로를 파이프라인에 직접 넘기지 말고 torchaudio 라이브러리로 오디오를 먼저 읽은 뒤 파이프라인에 전달하면 이 문제를 완벽하게 피할 수 있습니다. 경고 메시지에서 추천하는 공식 우회 방법이기도 합니다.

# RTTM(Rich Transcription Time Marked)이란?

**RTTM**은 오디오 파일에서 **"누가 언제 말했는지"**에 대한 화자 분리(Speaker Diarization) 결과를 텍스트로 저장하는 **국제 표준 파일 형식**입니다. 
`pyannote.audio`를 비롯한 대부분의 음성 AI 연구와 시스템에서 정답지(Ground Truth)를 만들거나 분석 결과를 저장할 때 이 형식을 사용합니다.

## 📝 RTTM 파일의 생김새

RTTM 파일은 메모장으로 열어볼 수 있는 단순한 텍스트 파일이며, 각 줄이 하나의 음성 구간을 나타냅니다. 공백(Space)으로 띄워진 여러 개의 데이터로 구성됩니다.

```text
SPEAKER audio_file 1 5.23 2.10 <NA> <NA> speaker_1 <NA> <NA>
SPEAKER audio_file 1 7.50 1.85 <NA> <NA> speaker_2 <NA> <NA>
```

## 🔍 핵심 데이터 분석

한 줄에 총 9~10개의 데이터가 들어가지만, 화자 분리에서 실제로 의미 있게 쓰이는 항목은 4가지입니다. (나머지는 보통 `<NA>`로 채워집니다.)

| 순서 | 항목명 | 설명 | 예시 |
|---|---|---|---|
| **1** | **Type** | 데이터의 종류 (항상 `SPEAKER`로 고정) | `SPEAKER` |
| **2** | **File ID** | 분석한 오디오 파일의 이름 | `audio_file` |
| **4** | **Start Time** | 화자가 말하기 시작한 시간 (초 단위) | `5.23` |
| **5** | **Duration** | 화자가 말한 길이/지속 시간 (초 단위) | `2.10` |
| **8** | **Speaker ID** | 식별된 화자의 이름이나 번호 | `speaker_1` |

> **💡 예시 줄 해석**
> "audio_file 이라는 음성 파일에서, speaker_1 이라는 사람이, 5.23초부터 2.10초 동안 말했다"

---

## 💻 Pyannote 결과를 RTTM으로 저장하는 법

`pyannote.audio`를 돌려 나온 결과를 파일로 저장하고 싶다면, 파이썬 코드 몇 줄만 추가하시면 됩니다.

```python
# diarization은 pipeline(오디오)를 실행한 결과물입니다.
with open("result.rttm", "w") as rttm_file:
    diarization.write_rttm(rttm_file)
```

In [4]:
import librosa
import torch

# 1. librosa를 사용해 MP3 파일을 안전하게 로드합니다.
audio_path = "audio/싼기타_비싼기타.mp3"
y, sample_rate = librosa.load(audio_path, sr=None, mono=False)

if y.ndim == 1:
    y = y[None, :]

waveform = torch.tensor(y)
audio_in_memory = {"waveform": waveform, "sample_rate": sample_rate}

# 2. 메모리에 로드된 오디오 데이터를 파이프라인에 전달
output = pipeline(audio_in_memory)

# 3. RTTM 파일로 결과 저장
with open("audio/싼기타_비싼기타.rttm", "w", encoding='utf-8') as rttm:
    output.speaker_diarization.write_rttm(rttm)

print("RTTM 파일 저장 완료!")

c:\Users\user\Documents\GitHub\STUDY\chaewony\ch05\.venv\Lib\site-packages\pyannote\audio\utils\reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
c:\Users\user\Documents\GitHub\STUDY\chaewony\ch05\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


RTTM 파일 저장 완료!


#### 발생 에러: torchcodec is not available / WinError 127 / torchaudio backend error

원인: 윈도우 환경에서 Pyannote 내부의 오디오 해독기(torchcodec)와 기본 라이브러리(torchaudio)가 MP3 파일을 제대로 읽어 들이지 못하고 뻗어버리는 고질적 버그 발생.

해결: 에러가 나는 내부 코덱을 쓰지 않고, MP3 읽기에 특화된 librosa 라이브러리를 추가 설치하여 오디오를 미리 숫자(Tensor)로 변환(메모리 로드)한 뒤 파이프라인에 주입하는 우회로를 구축함.

#### 발생 에러: AttributeError: 'DiarizeOutput' object has no attribute 'write_rttm'

원인: Pyannote가 4.x로 올라오면서, 파이프라인의 결과물이 단순한 화자 기록지에서 복잡한 '종합 출력 객체'로 변경됨.

해결: 결과물 전체에 대고 저장하라고 명령하는 대신, output.speaker_diarization으로 필요한 데이터만 정확히 집어내어 RTTM 형식으로 추출함.

In [5]:
import pandas as pd

rttm_path = "audio/싼기타_비싼기타.rttm"

# 1. RTTM 파일 읽어오기 (안전성 강화)
df_rttm = pd.read_csv(
    rttm_path,
    sep=r'\s+',      # 핵심 수정: ' ' 대신 정규식 '\s+'를 쓰면 공백이 1개든 2개든 에러 없이 쪼개줍니다.
    header=None,
    names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4'],
    encoding='utf-8' # 앞서 저장할 때 사용한 utf-8을 명시해 줍니다.
)

# 2. 불필요한 껍데기 데이터(<NA>, type 등)를 버리고 알맹이만 남기기
df_clean = df_rttm[['speaker_id', 'start', 'duration']].copy()

# 3. (꿀팁) 시작 시간(start)에 말한 길이(duration)를 더해 '끝난 시간(end)' 열을 추가해 두면 나중에 매우 편합니다.
df_clean['end'] = df_clean['start'] + df_clean['duration']

# 4. 보기 편하게 열 순서 정렬 (누가 -> 언제부터 -> 언제까지 -> 얼마나)
df_clean = df_clean[['speaker_id', 'start', 'end', 'duration']]

display(df_clean)

,speaker_id,start,end,duration
0,SPEAKER_00,0.993,6.798,5.805
1,SPEAKER_00,7.405,11.388,3.983
2,SPEAKER_00,11.759,16.686,4.927
3,SPEAKER_00,17.210,27.875,10.665
4,SPEAKER_00,28.668,30.204,1.536
...,...,...,...,...
83,SPEAKER_01,414.481,417.451,2.970
84,SPEAKER_00,417.755,421.231,3.476
85,SPEAKER_01,423.644,424.420,0.776
86,SPEAKER_01,424.741,428.268,3.527


In [6]:
# start + duration을 end로 변환
df_rttm['end'] = df_rttm['start'] + df_rttm['duration']

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end
0,SPEAKER,waveform,1,0.993,5.805,NaN,NaN,SPEAKER_00,NaN,NaN,6.798
1,SPEAKER,waveform,1,7.405,3.983,NaN,NaN,SPEAKER_00,NaN,NaN,11.388
2,SPEAKER,waveform,1,11.759,4.927,NaN,NaN,SPEAKER_00,NaN,NaN,16.686
3,SPEAKER,waveform,1,17.210,10.665,NaN,NaN,SPEAKER_00,NaN,NaN,27.875
4,SPEAKER,waveform,1,28.668,1.536,NaN,NaN,SPEAKER_00,NaN,NaN,30.204
...,...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,waveform,1,414.481,2.970,NaN,NaN,SPEAKER_01,NaN,NaN,417.451
84,SPEAKER,waveform,1,417.755,3.476,NaN,NaN,SPEAKER_00,NaN,NaN,421.231
85,SPEAKER,waveform,1,423.644,0.776,NaN,NaN,SPEAKER_01,NaN,NaN,424.420
86,SPEAKER,waveform,1,424.741,3.527,NaN,NaN,SPEAKER_01,NaN,NaN,428.268


In [7]:
df_rttm["number"] = None	# number 열 만들고 None으로 초기화
df_rttm.at[0, "number"] = 0

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,waveform,1,0.993,5.805,NaN,NaN,SPEAKER_00,NaN,NaN,6.798,0
1,SPEAKER,waveform,1,7.405,3.983,NaN,NaN,SPEAKER_00,NaN,NaN,11.388,None
2,SPEAKER,waveform,1,11.759,4.927,NaN,NaN,SPEAKER_00,NaN,NaN,16.686,None
3,SPEAKER,waveform,1,17.210,10.665,NaN,NaN,SPEAKER_00,NaN,NaN,27.875,None
4,SPEAKER,waveform,1,28.668,1.536,NaN,NaN,SPEAKER_00,NaN,NaN,30.204,None
...,...,...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,waveform,1,414.481,2.970,NaN,NaN,SPEAKER_01,NaN,NaN,417.451,None
84,SPEAKER,waveform,1,417.755,3.476,NaN,NaN,SPEAKER_00,NaN,NaN,421.231,None
85,SPEAKER,waveform,1,423.644,0.776,NaN,NaN,SPEAKER_01,NaN,NaN,424.420,None
86,SPEAKER,waveform,1,424.741,3.527,NaN,NaN,SPEAKER_01,NaN,NaN,428.268,None


In [8]:
for i in range(1, len(df_rttm)):
    if df_rttm.at[i, "speaker_id"] != df_rttm.at[i-1, "speaker_id"]:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"] + 1
    else:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"]

display(df_rttm.head(10)) 

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,waveform,1,0.993,5.805,NaN,NaN,SPEAKER_00,NaN,NaN,6.798,0
1,SPEAKER,waveform,1,7.405,3.983,NaN,NaN,SPEAKER_00,NaN,NaN,11.388,0
2,SPEAKER,waveform,1,11.759,4.927,NaN,NaN,SPEAKER_00,NaN,NaN,16.686,0
3,SPEAKER,waveform,1,17.210,10.665,NaN,NaN,SPEAKER_00,NaN,NaN,27.875,0
4,SPEAKER,waveform,1,28.668,1.536,NaN,NaN,SPEAKER_00,NaN,NaN,30.204,0
5,SPEAKER,waveform,1,32.414,0.759,NaN,NaN,SPEAKER_01,NaN,NaN,33.173,1
6,SPEAKER,waveform,1,33.545,3.561,NaN,NaN,SPEAKER_01,NaN,NaN,37.106,1
7,SPEAKER,waveform,1,37.628,3.763,NaN,NaN,SPEAKER_01,NaN,NaN,41.391,1
8,SPEAKER,waveform,1,41.611,0.844,NaN,NaN,SPEAKER_00,NaN,NaN,42.455,2
9,SPEAKER,waveform,1,41.645,1.063,NaN,NaN,SPEAKER_01,NaN,NaN,42.708,3


In [9]:
df_rttm_grouped = df_rttm.groupby("number").agg(
    start=pd.NamedAgg(column='start', aggfunc='min'),
    end=pd.NamedAgg(column='end', aggfunc='max'),
    speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first')
)

display(df_rttm_grouped)

,start,end,speaker_id
number,,,
0,0.993,30.204,SPEAKER_00
1,32.414,41.391,SPEAKER_01
2,41.611,42.455,SPEAKER_00
3,41.645,42.708,SPEAKER_01
4,42.674,44.024,SPEAKER_00
5,45.813,67.109,SPEAKER_01
6,67.227,82.786,SPEAKER_00
7,84.659,102.564,SPEAKER_01
8,103.492,117.532,SPEAKER_00


In [10]:
df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]
df_rttm_grouped = df_rttm_grouped.reset_index(drop=True)
display(df_rttm_grouped)

,start,end,speaker_id,duration
0,0.993,30.204,SPEAKER_00,29.211
1,32.414,41.391,SPEAKER_01,8.977
2,41.611,42.455,SPEAKER_00,0.844
3,41.645,42.708,SPEAKER_01,1.063
4,42.674,44.024,SPEAKER_00,1.350
5,45.813,67.109,SPEAKER_01,21.296
6,67.227,82.786,SPEAKER_00,15.559
7,84.659,102.564,SPEAKER_01,17.905
8,103.492,117.532,SPEAKER_00,14.040
9,119.759,138.676,SPEAKER_01,18.917


In [11]:
df_rttm_grouped.to_csv(
    "싼기타_비싼기타_rttm.csv",
    sep=',',
    index=False
)

## 받아쓰기 하는 함수 whisper_tts 만들기

In [12]:
import os
import torch
import pandas as pd
import librosa  # 윈도우 오디오 코덱 에러 우회를 위해 추가
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline
from pyannote.audio import Pipeline as PyannotePipeline

# ffmpeg 경로 설정 (자신의 환경에 맞게 유지)
os.environ["PATH"] += os.pathsep + r"C:\ffmpeg\bin" 

def whisper_stt(
    audio_file_path: str,      
    output_file_path: str = "output.csv"
):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model_id = "openai/whisper-large-v3-turbo"

    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, torch_dtype=torch_dtype, 
        low_cpu_mem_usage=True, 
        use_safetensors=True
    )
    model.to(device)

    processor = AutoProcessor.from_pretrained(model_id)

    pipe = hf_pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype,
        device=device,
        return_timestamps=True,  # 청크별로 타임스탬프를 반환
        chunk_length_s=10,  # 입력 오디오를 10초씩 나누기
        stride_length_s=2,  # 2초씩 겹치도록 청크 나누기
    )

    # 윈도우 torchcodec 에러 방지를 위해 librosa로 오디오 로드
    print("Whisper STT: 오디오 파일 로딩 중...")
    audio_array, sampling_rate = librosa.load(audio_file_path, sr=16000)
    audio_input = {"raw": audio_array, "sampling_rate": sampling_rate}

    print("Whisper STT: 텍스트 추출 시작 (시간이 조금 걸릴 수 있습니다)...")
    result = pipe(audio_input)
    
    df = whisper_to_dataframe(result, output_file_path)

    return result, df


def whisper_to_dataframe(result, output_file_path):
    start_end_text = []

    for chunk in result["chunks"]:
        start = chunk["timestamp"][0]
        end = chunk["timestamp"][1]
        text = chunk["text"].strip()
        start_end_text.append([start, end, text])
        
    df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
    df.to_csv(output_file_path, index=False, sep="|", encoding='utf-8')
    
    return df


def speaker_diarization(
        audio_file_path: str,
        output_rttm_file_path: str,
        output_csv_file_path: str
    ):

    pipeline = PyannotePipeline.from_pretrained(
        "pyannote/speaker-diarization-community-1",
        token=HUGGING_FACE_TOKEN
    )

    # cuda가 사용 가능한 경우 cuda를 사용하도록 설정
    if torch.cuda.is_available():
        pipeline.to(torch.device("cuda"))
        print('Pyannote: CUDA is available')
    else:
        print('Pyannote: CUDA is not available')
        
    # librosa를 이용해 오디오 파일을 메모리로 먼저 로드 (윈도우 에러 방지)
    print("Pyannote: 오디오 파일 로딩 중...")
    y, sample_rate = librosa.load(audio_file_path, sr=None, mono=False)
    if y.ndim == 1:
        y = y[None, :]
    waveform = torch.tensor(y)
    audio_in_memory = {"waveform": waveform, "sample_rate": sample_rate}

    print("Pyannote: 화자 분리 시작...")
    # 파이프라인 실행
    diarization_result = pipeline(audio_in_memory)

    # RTTM 저장 시 화자 분리(speaker_diarization) 객체만 지정하여 저장
    with open(output_rttm_file_path, "w", encoding='utf-8') as rttm:
        diarization_result.speaker_diarization.write_rttm(rttm)

    # pandas dataframe으로 변환 시 안전한 띄어쓰기 정규식(sep=r'\s+') 적용
    df_rttm = pd.read_csv(
        output_rttm_file_path,      
        sep=r'\s+',                 
        header=None,                
        names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4'],
        encoding='utf-8'
    )
    
    df_rttm["end"] = df_rttm["start"] + df_rttm["duration"]

    # speaker_id를 기반으로 화자별로 구간을 나누기
    df_rttm["number"] = None
    df_rttm.at[0, "number"] = 0

    for i in range(1, len(df_rttm)):
        if df_rttm.at[i, "speaker_id"] != df_rttm.at[i-1, "speaker_id"]:
            df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"] + 1
        else:
            df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"]

    df_rttm_grouped = df_rttm.groupby("number").agg(
        start=pd.NamedAgg(column='start', aggfunc='min'),
        end=pd.NamedAgg(column='end', aggfunc='max'),
        speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first')
    )

    df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]

    df_rttm_grouped.to_csv(
        output_csv_file_path,
        index=False,    
        encoding='utf-8' 
    )
    return df_rttm_grouped


def stt_to_rttm(
        audio_file_path: str,
        stt_output_file_path: str,
        rttm_file_path: str,
        rttm_csv_file_path: str,
        final_output_csv_file_path: str
    ):

    # 1. Whisper로 STT 진행
    result, df_stt = whisper_stt(
        audio_file_path, 
        stt_output_file_path
    ) 

    # 2. Pyannote로 화자 분리 진행
    df_rttm = speaker_diarization(
        audio_file_path,
        rttm_file_path,
        rttm_csv_file_path
    ) 

    # 3. STT 텍스트와 RTTM 화자 정보를 시간 기반으로 병합
    print("STT 결과와 화자 분리 결과를 병합 중...")
    df_rttm["text"] = "" 

    for i_stt, row_stt in df_stt.iterrows(): 
        overlap_dict = {}
        for i_rttm, row_rttm in df_rttm.iterrows(): 
            overlap = max(0, min(row_stt["end"], row_rttm["end"]) - max(row_stt["start"], row_rttm["start"]))
            overlap_dict[i_rttm] = overlap
        
        max_overlap = max(overlap_dict.values())
        max_overlap_idx = max(overlap_dict, key=overlap_dict.get)

        if max_overlap > 0: 
            df_rttm.at[max_overlap_idx, "text"] += row_stt["text"] + "\n"

    # 4. 최종 결과 CSV 저장
    df_rttm.to_csv(
        final_output_csv_file_path,
        index=False,    
        sep='|',
        encoding='utf-8'
    )  
    return df_rttm


if __name__ == "__main__":
    audio_file_path = "audio/싼기타_비싼기타.mp3"           # 원본 오디오 파일
    stt_output_file_path = "audio/싼기타_비싼기타.csv"      # STT 결과 파일
    rttm_file_path = "audio/싼기타_비싼기타.rttm"           # 화자 분리 원본 파일
    rttm_csv_file_path = "audio/싼기타_비싼기타_rttm.csv"   # 화자 분리 CSV 파일
    final_csv_file_path = "audio/싼기타_비싼기타_final.csv" # 최종 결과 파일

    # 모든 과정(STT + Diarization + 병합)을 한 번에 실행
    print("=== 작업을 시작합니다 ===")
    df_rttm = stt_to_rttm(
        audio_file_path,
        stt_output_file_path,
        rttm_file_path,
        rttm_csv_file_path,
        final_csv_file_path
    )

    print("\n=== 최종 병합 결과 ===")
    print(df_rttm)

=== 작업을 시작합니다 ===


Device set to use cuda:0


Whisper STT: 오디오 파일 로딩 중...


c:\Users\user\Documents\GitHub\STUDY\chaewony\ch05\.venv\Lib\site-packages\transformers\models\whisper\generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


Whisper STT: 텍스트 추출 시작 (시간이 조금 걸릴 수 있습니다)...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
c:\Users\user\Documents\GitHub\STUDY\chaewony\ch05\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--pyannote--speaker-diarization-community-1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` enviro

Pyannote: CUDA is available
Pyannote: 오디오 파일 로딩 중...
Pyannote: 화자 분리 시작...


c:\Users\user\Documents\GitHub\STUDY\chaewony\ch05\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


STT 결과와 화자 분리 결과를 병합 중...

=== 최종 병합 결과 ===
          start      end  speaker_id  duration  \
number                                           
0         0.993   30.204  SPEAKER_00    29.211   
1        32.414   41.391  SPEAKER_01     8.977   
2        41.611   42.455  SPEAKER_00     0.844   
3        41.645   42.708  SPEAKER_01     1.063   
4        42.674   44.024  SPEAKER_00     1.350   
5        45.813   67.109  SPEAKER_01    21.296   
6        67.227   82.786  SPEAKER_00    15.559   
7        84.659  102.564  SPEAKER_01    17.905   
8       103.492  117.532  SPEAKER_00    14.040   
9       119.759  138.676  SPEAKER_01    18.917   
10      139.351  168.967  SPEAKER_00    29.616   
11      170.907  192.321  SPEAKER_01    21.414   
12      192.322  193.689  SPEAKER_00     1.367   
13      192.760  193.503  SPEAKER_01     0.743   
14      193.823  216.571  SPEAKER_00    22.748   
15      218.579  237.783  SPEAKER_01    19.204   
16      238.103  238.677  SPEAKER_00     0.574   
17    

윈도우 dataset 읽어오는 버그

``` txt
---------------------------------------------------------------------------
OSError                                   Traceback (most recent call last)
Cell In[21], line 195
    191     final_csv_file_path = "audio/싼기타_비싼기타_final.csv" # 최종 결과 파일
    192 
    193     # 모든 과정(STT + Diarization + 병합)을 한 번에 실행
    194     print("=== 작업을 시작합니다 ===")
--> 195     df_rttm = stt_to_rttm(
    196         audio_file_path,
    197         stt_output_file_path,
    198         rttm_file_path,

Cell In[21], line 148
    144         final_output_csv_file_path: str
    145     ):
    146 
    147     # 1. Whisper로 STT 진행
--> 148     result, df_stt = whisper_stt(
    149         audio_file_path,
    150         stt_output_file_path
    151     )

Cell In[21], line 46
     42     audio_array, sampling_rate = librosa.load(audio_file_path, sr=16000)
     43     audio_input = {"raw": audio_array, "sampling_rate": sampling_rate}
...
--> 379     self._handle = _dlopen(self._name, mode)
    380 else:
    381     self._handle = handle

```

uv add "transformers<=4.48.3" "datasets<4.0.0"

```
왜 지금 고장났을까?
데이터셋 4.x 버전의 동작 변경 사항 : 오디오는 TorchCodec + FFmpeg을 통해 접근 시 디코딩됩니다. 이전 버전인 3.x에서는 다른 백엔드를 사용했습니다. 출력 예제를 통해 디코딩할 수 있습니다. ( Hugging Face )
새로운 런타임 요구 사항 : TorchCodec은 시스템에 FFmpeg이 설치되어 있고 호환되는 torch버전이어야 합니다. README 파일에는 FFmpeg 지원 및 torch↔torchcodec 비교표가 설명되어 있습니다. ( GitHub )
Windows 관련 주의사항 : 초기 4.0 릴리스 노트에는 "아직 Windows에서 사용할 수 없습니다. 4.0 미만 버전의 데이터셋을 사용하십시오."라는 경고가 있었습니다. 이 때문에 이전에 정상적으로 작동하던 Windows 설정이 업그레이드 후 오류를 일으키기 시작한 것입니다. ( GitHub )
```
https://discuss.huggingface.co/t/issue-with-torchcodec-when-fine-tuning-whisper-asr-model/169315